# NaviTrace LLaVA‑v1.6 Fine-Tuning Notebook

This notebook finetunes the **LLaVA‑v1.6** model on the NaviTrace dataset and compares several parameter-efficient fine-tuning strategies:

- **BitFit** (bias-only updates)
- **IA3** as the **adapter-style** baseline
- **LoRA**
- **QLoRA**

The notebook uses the following high-level stages:

1. Load and filter the NaviTrace data
2. Preprocess images and traces
3. Convert each sample into a multimodal instruction-tuning example
4. Fine-tune LLaVA‑v1.6 with multiple PEFT methods
5. Generate validation/test predictions
6. Evaluate with the same navigation metrics used in the original notebook
7. Compare methods in a summary table

## References used for the notebook design

For the multimodal fine-tuning path, this notebook follows Hugging Face guidance that:
- LLaVA‑NeXT / LLaVA‑v1.6 is supported in Transformers and can be loaded with an `AutoProcessor` and image-text generation model class.
- TRL supports **vision-language supervised fine-tuning** with `SFTTrainer`, custom image-text collation, and optional PEFT integration.
- PEFT officially supports adapter-style and LoRA-style methods such as **IA3** and **LoRA**. 
- QLoRA uses a frozen **4-bit quantized** base model together with LoRA adapters, with `nf4` recommended for 4-bit training.
- Classic bottleneck adapters are documented by AdapterHub.


In [ ]:
# Main environment setup
# Comment: install only what you need in your environment.
# The original notebook used CLIP-specific installs; this notebook switches to LLaVA/TRL/PEFT.

# !pip install -U datasets huggingface_hub torch torchvision Pillow numpy pandas scipy scikit-image tqdm matplotlib
# !pip install -U transformers accelerate trl peft bitsandbytes tensorboard sentencepiece
# Optional:
# !pip install -U hf_xet

import os
import io
import re
import ast
import json
import math
import time
import copy
import random
import pathlib
import functools
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import io
import base64

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from scipy.spatial import KDTree
from skimage.draw import line_aa
from skimage.draw import line as sk_line

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from peft import (
    LoraConfig,
    IA3Config,
    get_peft_model,
    prepare_model_for_kbit_training,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
print(f"bf16 support : {torch.cuda.is_available() and torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")


In [ ]:
# Global configuration

CFG = {
    # Data / task settings
    "embodiment": "Legged Robot",
    "val_split": 0.20,
    "seed": 42,

    # Quick experiment testing:
    "max_train_samples": None,      
    "max_val_samples": None,
    "max_test_samples": None,

    # Model settings:
    "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
    "image_max_side": 672,           
    "max_new_tokens": 256,
    "generation_num_beams": 1,

    # Training settings
    "num_train_epochs": 100,
    "learning_rate": 2e-5,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.03,
    "patience": 10, # Stopping criteria if val loss does not improve after n epochs
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_total_limit": 1,
    "gradient_checkpointing": True,
    "report_to": "tensorboard",

    # Formatting / output settings
    "trace_points": None,            
    "output_root": "./llava_navitrace_runs",
    "cache_root": "./llava_navitrace_cache",
    "penalty_tsv": "./category_penalty.tsv",
    "m2f_config": "./mask2former_config.json",
    "bad_score_threshold": 3234.75,
    "penalty_dist_thr": 35,

    # Method-specific default ranks
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}

os.makedirs(CFG["output_root"], exist_ok=True)
os.makedirs(CFG["cache_root"], exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

print(CFG)


## Data setup

This section follows the same overall logic as the original training notebook:
- load `leggedrobotics/navitrace`
- keep only samples with valid ground-truth traces for the requested embodiment
- compute a fixed train/validation split
- compute the median number of trace waypoints so all traces can be resampled to a common length


In [ ]:
# Load NaviTrace and reproduce the original train/val filtering logic
# Main step comment: keep only samples that have at least one GT trace for the chosen embodiment.

ALL_SAMPLES_PKL = os.path.join(CFG["cache_root"], "samples_all.pkl")

if os.path.exists(ALL_SAMPLES_PKL):
    print("Loading cached filtered samples...")
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = torch.load(f, weights_only=False) if ALL_SAMPLES_PKL.endswith(".pt") else None

if os.path.exists(ALL_SAMPLES_PKL) and saved is None:
    # Fall back to pickle if the cache was created as a pickle file in a prior run
    import pickle
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = pickle.load(f)

if os.path.exists(ALL_SAMPLES_PKL):
    val_split_samples = saved["val_split"]
    print(f"NaviTrace filtered validation split : {len(val_split_samples)} samples")
else:
    print("Loading NaviTrace from Hugging Face...")
    dataset = load_dataset("leggedrobotics/navitrace")
    print(f"Available splits: {list(dataset.keys())}")

    val_split_samples = []
    skipped = 0
    for s in tqdm(list(dataset["validation"]), desc="Filtering validation"):
        gt = s["ground_truth"].get(CFG["embodiment"])
        if gt is not None and len(gt) > 0:
            val_split_samples.append(s)
        else:
            skipped += 1

    print(f"Kept={len(val_split_samples)}  Skipped={skipped}")

    import pickle
    with open(ALL_SAMPLES_PKL, "wb") as f:
        pickle.dump({"val_split": val_split_samples}, f)

# Compute a common number of waypoints from the median GT trace length
trace_lengths = []
for s in val_split_samples:
    for t in s["ground_truth"][CFG["embodiment"]]:
        trace_lengths.append(len(t))

CFG["trace_points"] = int(np.median(trace_lengths))
print(f"Common trace length N = {CFG['trace_points']}")

# Fixed split for reproducibility
random.seed(CFG["seed"])
random.shuffle(val_split_samples)

n_our_val = int(len(val_split_samples) * CFG["val_split"])
val_samples = val_split_samples[:n_our_val]
train_samples = val_split_samples[n_our_val:]

if CFG["max_train_samples"] is not None:
    train_samples = train_samples[:CFG["max_train_samples"]]
if CFG["max_val_samples"] is not None:
    val_samples = val_samples[:CFG["max_val_samples"]]

print(f"Train samples: {len(train_samples)}")
print(f"Val samples  : {len(val_samples)}")


In [ ]:
# Load the test split for later leaderboard-style export
# Main step comment: keep only test samples that support the requested embodiment.

dataset = load_dataset("leggedrobotics/navitrace")

test_samples = []
for s in tqdm(list(dataset["test"]), desc="Filtering test"):
    if CFG["embodiment"] in s.get("embodiments", []):
        test_samples.append(s)

if CFG["max_test_samples"] is not None:
    test_samples = test_samples[:CFG["max_test_samples"]]

print(f"Test samples: {len(test_samples)}")


## Data Preprocessing 

- pad images to square
- map traces into padded normalized coordinates
- resample traces to a fixed number of waypoints
- convert normalized trace predictions back to pixel coordinates


In [ ]:
def pad_to_square(image: Image.Image):
    w, h = image.size
    max_s = max(w, h)
    padded = Image.new("RGB", (max_s, max_s), (0, 0, 0))
    x_off = (max_s - w) // 2
    y_off = (max_s - h) // 2
    padded.paste(image, (x_off, y_off))
    return padded, (x_off / max_s, y_off / max_s, w / max_s, h / max_s)

def resize_if_needed(image: Image.Image, max_side: int):
    w, h = image.size
    longest = max(w, h)
    if longest <= max_side:
        return image
    scale = max_side / longest
    return image.resize((int(round(w * scale)), int(round(h * scale))), Image.BICUBIC)

def adjust_trace(trace_px, orig_w, orig_h, xof, yof, sx, sy):
    t = np.array(trace_px, dtype=float)
    if t.ndim != 2 or t.shape[1] != 2:
        raise ValueError("Trace must have shape (num_points, 2)")
    t[:, 0] = (t[:, 0] / orig_w) * sx + xof
    t[:, 1] = (t[:, 1] / orig_h) * sy + yof
    return t.astype(np.float32)

def interpolate_trace(trace: np.ndarray, N: int) -> np.ndarray:
    if len(trace) == 0:
        return np.zeros((N, 2), dtype=np.float32)
    if len(trace) == 1:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    diffs = np.diff(trace, axis=0)
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    cumlen = np.concatenate([[0], np.cumsum(dists)])
    total = cumlen[-1]

    if total == 0:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    t_old = cumlen / total
    t_new = np.linspace(0, 1, N)
    out = np.stack(
        [
            np.interp(t_new, t_old, trace[:, 0]),
            np.interp(t_new, t_old, trace[:, 1]),
        ],
        axis=1,
    )
    return out.astype(np.float32)

def pred_padded_to_pixel(pred_norm, orig_w, orig_h):
    pred_norm = np.array(pred_norm, dtype=float)
    max_s = max(orig_w, orig_h)
    x_off = (max_s - orig_w) / 2.0
    y_off = (max_s - orig_h) / 2.0

    px = pred_norm.copy()
    px[:, 0] = px[:, 0] * max_s - x_off
    px[:, 1] = px[:, 1] * max_s - y_off

    px[:, 0] = np.clip(px[:, 0], 0, orig_w - 1)
    px[:, 1] = np.clip(px[:, 1], 0, orig_h - 1)
    return px.astype(np.float32)

def choose_canonical_trace(sample, N):
    # Main step comment: for generative supervision we need a single target sequence.
    gt_root = sample.get("ground_truth", None)
    if not isinstance(gt_root, dict):
        raise ValueError("Sample has no usable ground_truth dictionary")

    gt_traces = gt_root.get(CFG["embodiment"])
    if gt_traces is None or len(gt_traces) == 0:
        raise ValueError("Sample has no GT trace for the requested embodiment")

    gt_px = np.array(gt_traces[0], dtype=float)
    img = sample["image"]
    orig_w, orig_h = img.size
    _, (xof, yof, sx, sy) = pad_to_square(img)
    gt_norm = adjust_trace(gt_px, orig_w, orig_h, xof, yof, sx, sy)
    return interpolate_trace(gt_norm, N)

def compact_trace_json(trace_norm: np.ndarray, decimals: int = 4) -> str:
    # Main step comment: serialize the target trace as compact JSON for supervised generation.
    rounded = [[round(float(x), decimals), round(float(y), decimals)] for x, y in trace_norm]
    payload = {
        "goal": rounded[-1],
        "trace": rounded,
    }
    return json.dumps(payload, separators=(",", ":"))

def extract_first_json_object(text: str):
    # Main step comment: recover the model output as JSON even when generation contains extra text.
    if text is None:
        return None
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                snippet = text[start:i + 1]
                try:
                    return json.loads(snippet)
                except Exception:
                    return None
    return None

def json_to_trace_norm(obj, N):
    # Main step comment: safely convert model JSON back to a fixed-length normalized trace.

    if not isinstance(obj, dict) or "trace" not in obj:
        return None

    raw_trace = obj["trace"]
    if not isinstance(raw_trace, (list, tuple)):
        return None

    points = []

    for item in raw_trace:
        # Standard [x, y]
        if isinstance(item, (list, tuple)) and len(item) == 2:
            a, b = item[0], item[1]

            # [x, y]
            if np.isscalar(a) and np.isscalar(b):
                try:
                    points.append([float(a), float(b)])
                    continue
                except (TypeError, ValueError):
                    pass

            # [[x, y], extra] or [extra, [x, y]] -> salvage the pair if present
            for candidate in (a, b):
                if isinstance(candidate, (list, tuple)) and len(candidate) == 2:
                    try:
                        points.append([float(candidate[0]), float(candidate[1])])
                        break
                    except (TypeError, ValueError):
                        pass
            else:
                pass
            continue

        # Nested singleton like [[x, y]]
        if (
            isinstance(item, (list, tuple))
            and len(item) == 1
            and isinstance(item[0], (list, tuple))
            and len(item[0]) == 2
        ):
            try:
                points.append([float(item[0][0]), float(item[0][1])])
                continue
            except (TypeError, ValueError):
                pass

        # Dict point {"x": ..., "y": ...}
        if isinstance(item, dict) and "x" in item and "y" in item:
            try:
                points.append([float(item["x"]), float(item["y"])])
                continue
            except (TypeError, ValueError):
                pass

        # Flat 2-number dict variants
        if isinstance(item, dict) and "point" in item:
            p = item["point"]
            if isinstance(p, (list, tuple)) and len(p) == 2:
                try:
                    points.append([float(p[0]), float(p[1])])
                    continue
                except (TypeError, ValueError):
                    pass

    if len(points) == 0:
        return None

    trace = np.asarray(points, dtype=np.float32)

    if trace.ndim != 2 or trace.shape[1] != 2:
        return None

    trace = np.clip(trace, 0.0, 1.0)

    if len(trace) == N:
        return trace.tolist()

    if len(trace) == 1:
        return np.repeat(trace, N, axis=0).tolist()

    old_idx = np.linspace(0.0, 1.0, len(trace))
    new_idx = np.linspace(0.0, 1.0, N)
    x_new = np.interp(new_idx, old_idx, trace[:, 0])
    y_new = np.interp(new_idx, old_idx, trace[:, 1])

    return np.stack([x_new, y_new], axis=1).tolist()


In [ ]:
# import inspect
# print(inspect.getsource(json_to_trace_norm))

## Build multimodal instruction-tuning examples
  
For LLaVA, each sample is turned into a **conversation**:

- **system**: explain the output format
- **user**: provide the image and task text
- **assistant**: return the target trajectory JSON

This lets us fine-tune LLaVA as a vision-language sequence model while still preserving the navigation objective.


In [ ]:
SYSTEM_PROMPT = (
    "You are a visual navigation model for the NaviTrace benchmark. "
    "Given a scene image and a navigation instruction, predict a feasible 2D path for the "
    "Legged Robot as normalized waypoints in the padded-square image frame. "
    "Return JSON only with keys 'goal' and 'trace'. "
    "'goal' must be the last waypoint [x, y]. "
    "'trace' must be an ordered list of [x, y] pairs with values in [0, 1]. "
    "Do not add explanations."
)

def get_sample_id(sample):
    return (
        sample.get("sample_id")
        or sample.get("id")
        or sample.get("uid")
        or "unknown"
    )

def get_sample_task(sample):
    for key in ["task", "instruction", "text", "prompt", "command", "query", "description"]:
        val = sample.get(key, None)
        if isinstance(val, str) and val.strip():
            return val
    raise KeyError(f"No task/instruction field found. Keys: {list(sample.keys())}")

def get_sample_category(sample):
    return sample.get("category", sample.get("scene_category", "unknown"))

def make_messages_for_sample(sample, include_answer=True):
    # Main step comment: package each sample as a multimodal chat conversation.
    img = sample["image"].convert("RGB")
    img = resize_if_needed(img, CFG["image_max_side"])

    task_text = get_sample_task(sample)

    user_text = (
        f"Task: {task_text}\n"
        f"Embodiment: {CFG['embodiment']}\n"
        f"Predict the path as JSON only."
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ],
        },
    ]

    if include_answer:
        trace_norm = choose_canonical_trace(sample, CFG["trace_points"])
        answer = compact_trace_json(trace_norm)
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        )

    return messages


def build_sft_dataset(samples, include_answer=True):
    rows = []
    for s in tqdm(samples, desc="Formatting messages"):
        try:
            rows.append(
                {
                    "sample_id": s["sample_id"],
                    "task": s["task"],
                    "category": s["category"],
                    "messages": make_messages_for_sample(s, include_answer=include_answer),
                }
            )
        except Exception as e:
            print(f"Skipping sample {s.get('sample_id', 'unknown')} due to formatting error: {e}")
    return Dataset.from_list(rows)


print("train sample keys:", train_samples[0].keys())
print("val sample keys:", val_samples[0].keys())
print("test sample keys:", test_samples[0].keys())

print(make_messages_for_sample(train_samples[0], include_answer=True))
print(make_messages_for_sample(test_samples[0], include_answer=False))

train_ds = build_sft_dataset(train_samples, include_answer=True)
val_ds = build_sft_dataset(val_samples, include_answer=True)
test_prompt_ds = build_sft_dataset(test_samples, include_answer=False)

train_ds, val_ds, test_prompt_ds


In [ ]:
# Inspect one formatted training example
# Main step comment: confirm the final multimodal chat structure before training.

example = train_ds[0]
example["messages"]

In [ ]:
# Testing images
example = train_ds[0]
for i, msg in enumerate(example["messages"]):
    print(f"\nMessage {i} role={msg['role']}")
    for j, item in enumerate(msg["content"]):
        print(
            f"  item {j}: type={item.get('type')}, "
            f"has_image_key={'image' in item}, "
            f"image_is_none={item.get('image', 'MISSING') is None}"
        )

## Load processor

LLaVA‑v1.6 is used through the Hugging Face processor so the same notebook can support training and generation. The LLaVA model card recommends using the chat template through the processor.


In [ ]:
processor = AutoProcessor.from_pretrained(CFG["model_id"])
processor.tokenizer.padding_side = "right"

# Some checkpoints may require these fields to be present on the processor for image-token expansion.
# They are mentioned in HF discussions for LLaVA-NeXT processors.
if not hasattr(processor, "patch_size") and hasattr(processor, "image_processor"):
    processor.patch_size = getattr(processor.image_processor, "patch_size", None)

if not hasattr(processor, "vision_feature_select_strategy"):
    processor.vision_feature_select_strategy = "default"

print(type(processor))
print(processor.tokenizer.__class__.__name__)


## Training utilities

The common training loop is shared; only the model-loading and trainable-parameter setup changes.


In [ ]:
METHODS = {
    "bitfit": {
        "description": "Bias-only fine-tuning (BitFit)",
    },
    "ia3_adapter": {
        "description": "Adapter-style IA3 via PEFT",
    },
    "lora": {
        "description": "Standard LoRA on all linear layers",
    },
    "qlora": {
        "description": "4-bit quantized base model + LoRA (QLoRA)",
    },
}

METHODS


In [ ]:
def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100.0 * trainable / total if total > 0 else 0.0
    return {"trainable": trainable, "total": total, "pct": pct}

def print_trainable_summary(model, title="Model"):
    stats = count_trainable_parameters(model)
    print(
        f"{title}: trainable={stats['trainable']:,} / total={stats['total']:,} "
        f"({stats['pct']:.4f}%)"
    )

def set_bitfit_trainable(model):
    # Main step comment: BitFit trains only bias parameters.
    for p in model.parameters():
        p.requires_grad = False

    for name, p in model.named_parameters():
        if name.endswith(".bias") or name == "bias":
            p.requires_grad = True

    return model

def load_base_model_for_method(method_name: str):
    # Main step comment: load the correct base model depending on whether quantization is needed.
    common_kwargs = {
        "torch_dtype": DTYPE,
        "low_cpu_mem_usage": True,
        "device_map": "auto" if torch.cuda.is_available() else None,
    }

    if method_name == "qlora":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            bnb_4bit_quant_storage=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        model = AutoModelForImageTextToText.from_pretrained(
            CFG["model_id"],
            quantization_config=quant_config,
            **common_kwargs,
        )
        model = prepare_model_for_kbit_training(model)
        return model

    model = AutoModelForImageTextToText.from_pretrained(
        CFG["model_id"],
        **common_kwargs,
    )
    return model

def apply_peft_or_bitfit(model, method_name: str):
    # Main step comment: wrap the model according to the chosen PEFT strategy.
    if method_name == "bitfit":
        model = set_bitfit_trainable(model)
        return model

    if method_name == "ia3_adapter":
        peft_cfg = IA3Config(
            task_type="CAUSAL_LM",
            target_modules=["k_proj", "v_proj", "down_proj"],
            feedforward_modules=["down_proj"],
        )
        model = get_peft_model(model, peft_cfg)
        return model

    if method_name in ("lora", "qlora"):
        peft_cfg = LoraConfig(
            r=CFG["lora_r"],
            lora_alpha=CFG["lora_alpha"],
            lora_dropout=CFG["lora_dropout"],
            bias="none",
            target_modules="all-linear",
            task_type="CAUSAL_LM",
            modules_to_save=["lm_head", "embed_tokens"],
        )
        model = get_peft_model(model, peft_cfg)
        return model

    raise ValueError(f"Unknown method: {method_name}")


# Safe image loading:

In [ ]:
def to_pil_image_safe(obj):
    if obj is None:
        return None

    if isinstance(obj, Image.Image):
        return obj.convert("RGB")

    if isinstance(obj, dict):
        if obj.get("bytes") is not None:
            return Image.open(io.BytesIO(obj["bytes"])).convert("RGB")
        if obj.get("path"):
            return Image.open(obj["path"]).convert("RGB")
        if obj.get("base64") is not None:
            return Image.open(io.BytesIO(base64.b64decode(obj["base64"]))).convert("RGB")

    raise TypeError(f"Unsupported image object type: {type(obj)}")


def gather_images_from_messages(messages):
    images = []
    for message in messages:
        content = message.get("content", [])
        if not isinstance(content, list):
            continue

        for item in content:
            if not isinstance(item, dict):
                continue

            if item.get("type") == "image" and item.get("image") is not None:
                pil_img = to_pil_image_safe(item["image"])
                if pil_img is not None:
                    images.append(pil_img)

    return images

def build_collate_fn(processor, train_on_assistant_only=False):
    # Main step comment: create a VLM batch that includes both the chat text and the image tensors.
    def collate_fn(examples):
        texts = [
            processor.apply_chat_template(
                ex["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            for ex in examples
        ]

        # Convert serialized images back to PIL and flatten single-image examples
        image_lists = [gather_images_from_messages(ex["messages"]) for ex in examples]

        images = []
        for img_list in image_lists:
            if len(img_list) == 0:
                raise ValueError("No image found in example messages")
            elif len(img_list) == 1:
                images.append(img_list[0])   # flat single-image case
            else:
                images.append(img_list)      # keep multi-image case if ever used

        batch = processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()

        if processor.tokenizer.pad_token_id is not None:
            labels[labels == processor.tokenizer.pad_token_id] = -100

        image_token_candidates = set()
        for tok_name in ["image_token", "boi_token", "eoi_token"]:
            tok = processor.tokenizer.special_tokens_map.get(tok_name, None)
            if tok is not None:
                try:
                    image_token_candidates.add(processor.tokenizer.convert_tokens_to_ids(tok))
                except Exception:
                    pass

        image_token_candidates.add(262144)

        for tok_id in image_token_candidates:
            labels[labels == tok_id] = -100

        if train_on_assistant_only:
            pass

        batch["labels"] = labels
        return batch

    return collate_fn


In [ ]:
class HistoryCallback(TrainerCallback):
    # Main step comment: collect train/eval losses so we can compare methods afterwards.
    def __init__(self):
        self.history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            row = {"step": state.global_step, **logs}
            self.history.append(row)

def make_training_args(method_name: str, output_dir: str):
    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16_ok = torch.cuda.is_available() and not bf16_ok

    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=CFG["num_train_epochs"],
        learning_rate=CFG["learning_rate"],
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        per_device_train_batch_size=CFG["per_device_train_batch_size"],
        per_device_eval_batch_size=CFG["per_device_eval_batch_size"],
        gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
        logging_steps=CFG["logging_steps"],
        save_strategy=CFG["save_strategy"],
        eval_strategy=CFG["eval_strategy"],
        save_total_limit=CFG["save_total_limit"],
        gradient_checkpointing=CFG["gradient_checkpointing"],
        remove_unused_columns=False,   # important for multimodal examples
        report_to=CFG["report_to"],
        bf16=bf16_ok,
        fp16=fp16_ok,
        dataloader_num_workers=0,
        load_best_model_at_end=False,
        metric_for_best_model=None,
    )

def train_one_method(method_name: str, train_ds: Dataset, val_ds: Dataset):
    # Main step comment: load the model, apply the chosen tuning strategy, train, and save logs.
    run_dir = os.path.join(CFG["output_root"], method_name)
    os.makedirs(run_dir, exist_ok=True)

    model = load_base_model_for_method(method_name)
    model = apply_peft_or_bitfit(model, method_name)

    if hasattr(model, "config"):
        model.config.use_cache = False

    print_trainable_summary(model, title=f"{method_name} model")

    collate_fn = build_collate_fn(processor)
    training_args = make_training_args(method_name, run_dir)
    history_cb = HistoryCallback()

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        callbacks=[history_cb],
    )

    start = time.time()
    trainer.train()
    train_time_sec = time.time() - start

    trainer.save_model(run_dir)
    processor.save_pretrained(run_dir)

    hist_df = pd.DataFrame(history_cb.history)
    hist_path = os.path.join(run_dir, "history.csv")
    hist_df.to_csv(hist_path, index=False)

    return {
        "method": method_name,
        "run_dir": run_dir,
        "trainer": trainer,
        "model": model,
        "history": hist_df,
        "train_time_sec": train_time_sec,
    }


## Evaluation helpers

- Semantic penalty
- FDE
- DTW
- Normalized trace score

Predictions are generated by LLaVA as JSON and then parsed back into waypoint arrays.

In [ ]:
@functools.lru_cache(maxsize=4)
def penalty_lookup(embodiment):
    df = pd.read_csv(CFG["penalty_tsv"], sep="\t")
    with open(CFG["m2f_config"]) as f:
        m2f = json.load(f)
    id2label = {int(k): v for k, v in m2f["id2label"].items()}
    lkp = {}
    for lid, name in id2label.items():
        row = df[df["category"] == name]
        lkp[lid] = float(row.iloc[0][embodiment]) * 0.8 if len(row) else 0.0
    return lkp

def rasterize(trace, H, W):
    pts, pixels = np.array(trace), []
    if len(pts) > 1:
        for i in range(len(pts) - 1):
            r0, c0 = int(round(pts[i][1])), int(round(pts[i][0]))
            r1, c1 = int(round(pts[i + 1][1])), int(round(pts[i + 1][0]))
            rr, cc, _ = line_aa(r0, c0, r1, c1)
            v = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
            pixels.extend(zip(rr[v], cc[v]))
    elif len(pts) == 1:
        r, c = int(round(pts[0][1])), int(round(pts[0][0]))
        if 0 <= r < H and 0 <= c < W:
            pixels.append((r, c))
    return np.array(pixels)

def penalty_mask(seg, gt_px, emb, dthr=35):
    H, W = seg.shape
    mask = np.zeros((H, W), dtype=float)
    lkp = penalty_lookup(emb)
    gtp = rasterize(gt_px, H, W)
    if len(gtp) == 0:
        return mask

    tree = KDTree(gtp)
    ids = seg.ravel()
    und = np.where(np.isin(ids, list(lkp.keys())))[0]
    if und.size == 0:
        return mask

    rows, cols = np.unravel_index(und, (H, W))
    coords = np.vstack((rows, cols)).T
    dist, _ = tree.query(coords)
    pen_idx = coords[dist > dthr]

    if pen_idx.size > 0:
        rp, cp = pen_idx[:, 0], pen_idx[:, 1]
        mask[rp, cp] = np.vectorize(lkp.get)(seg[rp, cp], 0)

    return mask

def sem_penalty(pred_px, pmask):
    H, W = pmask.shape
    vals = []
    for i in range(len(pred_px) - 1):
        x1, y1 = int(round(pred_px[i][0])), int(round(pred_px[i][1]))
        x2, y2 = int(round(pred_px[i + 1][0])), int(round(pred_px[i + 1][1]))
        rr, cc = sk_line(y1, x1, y2, x2)
        v = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
        vals.extend(pmask[rr[v], cc[v]].tolist())
    return float(np.mean(vals)) if vals else 0.0

def calc_fde(p, g):
    return float(np.linalg.norm(np.array(p[-1]) - np.array(g[-1])))

def calc_dtw(p, g):
    p = np.array(p, dtype=float)
    g = np.array(g, dtype=float)

    if len(p) != len(g):
        longer, shorter = (p, g) if len(p) >= len(g) else (g, p)
        d = np.cumsum([0] + [np.linalg.norm(shorter[i] - shorter[i - 1]) for i in range(1, len(shorter))])
        tot = d[-1]
        if tot > 0:
            d = d / tot
        t = np.linspace(0, 1, len(longer))
        shorter = np.column_stack(
            [
                np.interp(t, d, shorter[:, 0]),
                np.interp(t, d, shorter[:, 1]),
            ]
        )
        p, g = (longer, shorter) if len(p) >= len(g) else (shorter, longer)

    n, m = len(p), len(g)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = np.linalg.norm(p[i - 1] - g[j - 1]) + min(
                D[i - 1, j], D[i, j - 1], D[i - 1, j - 1]
            )
    return float(D[n, m])

def norm_score(raw):
    return (CFG["bad_score_threshold"] - raw) / CFG["bad_score_threshold"] * 100.0


In [ ]:
def get_sample_instruction(sample):
    # Main step comment: retrieve the text instruction from any supported dataset schema.
    candidate_keys = [
        "instruction",
        "text",
        "prompt",
        "command",
        "query",
        "description",
        "goal",
        "task",
    ]

    for k in candidate_keys:
        v = sample.get(k, None)
        if isinstance(v, str) and v.strip():
            return v

    raise KeyError(
        f"No instruction-like field found. Available keys: {list(sample.keys())}"
    )

@torch.no_grad()
def generate_trace_json_val(model, sample):
    # Main step comment: run autoregressive multimodal generation for one sample.
    prompt_messages = make_messages_for_sample(sample, include_answer=False)
    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    images = gather_images_from_messages(prompt_messages)

    inputs = processor(
        text=[prompt_text],
        images=[images],
        return_tensors="pt",
        padding=True,
    )

    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    generated = model.generate(
        **inputs,
        max_new_tokens=CFG["max_new_tokens"],
        num_beams=CFG["generation_num_beams"],
        do_sample=False,
    )

    prompt_len = inputs["input_ids"].shape[1]
    completion_ids = generated[:, prompt_len:]
    text = processor.batch_decode(completion_ids, skip_special_tokens=True)[0]
    return text

@torch.no_grad()
def generate_trace_json_test(model, sample):
    # Main step comment: generate a JSON waypoint trace from one test sample.

    image = sample["image"]
    instruction = get_sample_instruction(sample)

    prompt = (
        f"You are predicting a normalized navigation trace for a {CFG['embodiment']}.\n"
        f"Given the image and instruction, output JSON only in the format:\n"
        f'{{"trace": [[x1, y1], [x2, y2], ..., [{CFG["trace_points"]} points total]]}}\n'
        f"Instruction: {instruction}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

    gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
    gen_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return gen_text

def evaluate_method_on_val(model, samples):
    # Main step comment: generate validation predictions and score them with the original navigation metrics.
    nt_scores = []
    dtw_scores = []
    fde_scores = []
    pen_scores = []
    invalid_json = 0
    raw_generations = []

    for sample in tqdm(samples, desc="Evaluating val"):
        gen_text = generate_trace_json_val(model, sample)
        raw_generations.append(
            {
                "sample_id": sample["sample_id"],
                "text": gen_text,
            }
        )

        payload = extract_first_json_object(gen_text)
        try:
            pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
        except Exception:
            print("\nBad payload:")
            print(payload)
            raise

        if pred_norm is None:
            invalid_json += 1
            # Fallback: use a degenerate center trace so the sample can still be scored
            pred_norm = np.tile(np.array([[0.5, 0.5]], dtype=np.float32), (CFG["trace_points"], 1))

        img = sample["image"]
        orig_w, orig_h = img.size
        seg = np.array(sample["segmentation_mask"])
        trace_px = pred_padded_to_pixel(pred_norm, orig_w, orig_h)

        gt_traces = sample["ground_truth"].get(CFG["embodiment"])
        best_raw = best_d = best_f = best_p = float("inf")

        for gt_px in gt_traces:
            pmask = penalty_mask(seg, gt_px, CFG["embodiment"], CFG["penalty_dist_thr"])
            p_val = sem_penalty(trace_px, pmask)
            f_val = calc_fde(trace_px, gt_px)
            d_val = calc_dtw(trace_px, gt_px)
            raw = d_val + f_val + p_val
            if raw < best_raw:
                best_raw, best_d, best_f, best_p = raw, d_val, f_val, p_val

        nt_scores.append(norm_score(best_raw))
        dtw_scores.append(best_d)
        fde_scores.append(best_f)
        pen_scores.append(best_p)

    metrics = {
        "trace_score_mean": float(np.mean(nt_scores)),
        "trace_score_std": float(np.std(nt_scores)),
        "dtw_mean": float(np.mean(dtw_scores)),
        "fde_mean": float(np.mean(fde_scores)),
        "penalty_mean": float(np.mean(pen_scores)),
        "invalid_json": int(invalid_json),
        "num_scored": int(len(nt_scores)),
    }

    return metrics, pd.DataFrame(raw_generations)

def export_test_predictions(model, samples, out_tsv):
    # Main step comment: produce leaderboard-style TSV output for the test split.
    rows = []

    for sample in tqdm(samples, desc="Generating test predictions"):
        try:
            gen_text = generate_trace_json_test(model, sample)
            payload = extract_first_json_object(gen_text)
            pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
        except Exception as e:
            print(f"Warning: failed on sample {sample.get('id', 'unknown')}: {e}")
            pred_norm = [[0.0, 0.0] for _ in range(CFG["trace_points"])]

        rows.append({
            "sample_id": sample.get("id", ""),
            "prediction": json.dumps(pred_norm),
        })

    pd.DataFrame(rows).to_csv(out_tsv, sep="\t", index=False)
    print(f"Saved test predictions to: {out_tsv}")


QUICK TEST w/ Single Batch:

In [ ]:
tmp_collate = build_collate_fn(processor)
tmp_batch = tmp_collate([train_ds[0], train_ds[1]])
for k, v in tmp_batch.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

In [ ]:
# result = train_one_method("bitfit", train_ds, val_ds)
# sample = test_samples[0]
# print(sample.keys())
# print(sample.get("ground_truth", None))
# gen_text = generate_trace_json_test(result["model"], sample)
# print(gen_text)

## Run the experiments

This cell fine-tunes LLaVA‑v1.6 once for each method and then evaluates the trained model on the validation set.

In [ ]:
# Select which methods to run
# Main step comment: edit this list when doing smaller pilot runs.

# METHODS_TO_RUN = ["ia3_adapter", "lora", "qlora"]
# METHODS_TO_RUN = ["bitfit"]
METHODS_TO_RUN = ["lora", "qlora"]

experiment_outputs = []

for method_name in METHODS_TO_RUN:
    print("=" * 100)
    print(f"Running method: {method_name} -> {METHODS[method_name]['description']}")
    print("=" * 100)

    result = train_one_method(method_name, train_ds, val_ds)

    metrics, raw_gen_df = evaluate_method_on_val(result["model"], val_samples)
    raw_gen_path = os.path.join(result["run_dir"], "val_generations.csv")
    raw_gen_df.to_csv(raw_gen_path, index=False)

    test_tsv_path = os.path.join(result["run_dir"], f"test_predictions_{method_name}.tsv")
    export_test_predictions(result["model"], test_samples, test_tsv_path)

    summary_row = {
        "method": method_name,
        "description": METHODS[method_name]["description"],
        "run_dir": result["run_dir"],
        "train_time_sec": result["train_time_sec"],
        **metrics,
    }
    experiment_outputs.append(summary_row)

    # Clean up between methods to reduce GPU memory pressure
    del result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_outputs).sort_values("trace_score_mean", ascending=False)
results_df


If you need to stop the top block of training early before all methods finish, run this to get the results of the methods you've already completed:

In [ ]:
# results_df = pd.DataFrame(experiment_outputs).sort_values("trace_score_mean", ascending=False)
# results_df

# Get Results

In [ ]:

# Save the comparison table
# Main step comment: store the full method comparison for later analysis.

results_csv = os.path.join(CFG["output_root"], "llava_method_comparison.csv")
results_df.to_csv(results_csv, index=False)
print(results_csv)
results_df


In [ ]:
# Plot method comparison
# Main step comment: visually compare validation performance across BitFit / IA3 / LoRA / QLoRA.

if len(results_df) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(results_df["method"], results_df["trace_score_mean"])
    plt.ylabel("Validation Trace Score")
    plt.title("LLaVA-v1.6 Fine-Tuning Method Comparison")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# plot training / eval loss curves for each method from the saved history.csv files

for method_name in METHODS_TO_RUN:
    hist_path = os.path.join(CFG["output_root"], method_name, "history.csv")
    if not os.path.exists(hist_path):
        continue

    hist = pd.read_csv(hist_path)
    if hist.empty:
        continue

    plt.figure(figsize=(8, 4))
    if "loss" in hist.columns:
        train_hist = hist[hist["loss"].notna()]
        plt.plot(train_hist["step"], train_hist["loss"], label="train")
    if "eval_loss" in hist.columns:
        eval_hist = hist[hist["eval_loss"].notna()]
        plt.plot(eval_hist["step"], eval_hist["eval_loss"], label="val")

    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"Training curve ({method_name})")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## Practical notes

1. **Resource requirements**  
   LLaVA‑v1.6 is much heavier than the original CLIP baseline. In practice:
   - **BitFit** and **LoRA** still need substantial memory.
   - **QLoRA** is usually the most practical starting point on a single GPU. citeturn1search2turn1search5

2. **Adapter experiment**  
   This notebook uses **IA3** as the adapter-style PEFT method because it is officially supported in the Hugging Face PEFT stack. 

3. **Target format**  
   The model is trained to **generate a JSON trace**, which is then parsed and scored with the downstream navigation metrics.

4. **Multiple GT traces**  
   For generative supervision, this notebook uses the **first GT trace** as the canonical training target, but validation still scores against **all available GT traces** and keeps the best match.

5. **Quick debugging mode**  
   Before full runs, set:
   ```python
   CFG["max_train_samples"] = 32
   CFG["max_val_samples"] = 8
   CFG["max_test_samples"] = 8
   CFG["num_train_epochs"] = 1
   METHODS_TO_RUN = ["qlora"]
   ```
